Urban Data Science & Smart Cities <br>
URSP688Y Spring 2026<br>
Instructor: Chester Harvey <br>
Urban Studies & Planning <br>
National Center for Smart Growth <br>
University of Maryland

# Exercise04

This last exercise is an opportunity for you to get started on your final project. Please identify a portion of your project to get started on and submit a notebook (and any other related files) where you:

1. State the question you are aiming to address with this portion of your analysis
2. Outline the approach you will use to answer that question (pseudocode or you can start to more formally outline the approach section for your final narrative)
3. Operationalize your approach with data and code that you can later slot into your final analysis

## Submitting

Please make a pull request with all of your code and reasonably-sized data in a folder named with your first name. See my example file structure in the `exercise04` directory:

```
exercises
└── exercise04
    └── chester
        ├── example_ReadMe.md
        ├── exercise04_chester.ipynb
        ├── example_module.py
        └── example_data.csv
```

If you have datasets that are too large for GitHub or should not be made public, please upload them to the [Google Drive folder for final project data](https://drive.google.com/drive/folders/1jgLIxx1B67nD4PekfDJnUI5jmrqa4iZm). Please also provide instructions for how someone running your code should properly locate or connect to these files so the analysis will run properly. For example, should they copy and paste the files into the same directory as your notebook, or a provided `data` directory? Best practice is to include these instructions in a separate ReadMe.md or ReadMe.txt file, or at the top of your notebook.

In [ ]:
import pandas as pd
import geopandas as gpd

## Prepare property data

In [ ]:
#skip this block#
#Import affordable housing poperties data

properties_df = pd.read_excel("Active_and_Inconclusive_Properties.xlsx")


In [ ]:
#skip this block#
#save as csv for quicker import 
properties_df.to_csv("affordable_properties.csv", index=False)

In [ ]:
#import data from csv
properties_df = pd.read_csv("affordable_properties.csv", low_memory=False)

In [ ]:
#select columns 
properties_df = properties_df[[
    "PropertyName",
    "PropertyAddress",
    "City",
    "State",
    "Zip",
    "County",
    "CensusTract",
    "Latitude",
    "Longitude",
    "PropertyStatus",
    "ActiveSubsidies",
    "TotalUnits",
    "EarliestEndDate",
    "LatestEndDate",
    "OwnerType",
    "PercentofELIHouseholds",
    "NumberActiveSection8",
    "NumberActiveSection202",
    "NumberActiveSection236",
    "NumberActiveHUDInsured",
    "NumberActiveLihtc",
    "NumberActiveSection515",
    "NumberActiveSection538",
    "NumberActiveHome",
    "NumberActivePublicHousing",
    "NumberActiveState",
    "NumberActivePBV",
    "NumberActiveMR",
    "NumberActiveNHTF"
]]

In [ ]:
#Filter only properties in PG County #no missing values found 
properties_df = properties_df[properties_df["State"] == "MD"]

properties_df = properties_df[properties_df["County"] == "Prince Georges"]


In [ ]:
#reset index
properties_df = properties_df.reset_index(drop=True)

In [ ]:
# input missing zip codes 
properties_df.loc[75, "Zip"] = 20737
properties_df.loc[110, "Zip"] = 20740

In [ ]:
#convert all zip codes to strings 
properties_df["Zip"] = properties_df["Zip"].astype(int).astype(str)

In [ ]:
#convert properties_df to a gdf

#create points
property_points = gpd.points_from_xy(properties_df["Longitude"], properties_df["Latitude"])

In [ ]:
#create gdf
properties_gdf = gpd.GeoDataFrame(properties_df, geometry=property_points, crs=4326)

In [ ]:
# Reproject into UTM 18 North, the local UTM zone (EPSG: 32618)
properties_gdf = properties_gdf.to_crs(32618)

## Prepare overlay

In [ ]:
#import 2024 acs data
acs2024_gdf = gpd.read_file("acs2024_5yr_B25064_14000US24033803606.geojson")

In [ ]:
# Reproject into UTM 18 North, the local UTM zone (EPSG: 32618)
acs2024_gdf = acs2024_gdf.to_crs(32618)

In [ ]:
#rename columns
acs2024_gdf = acs2024_gdf.rename(columns={"name": "census_tract","B25064001": "median_rent_2024", "B25064001, Error": "median_rent_2024_se"})

In [ ]:
#create a column with just the tract number
acs2024_gdf["tract_id"] = (acs2024_gdf["census_tract"].str.split().str[2].str.replace(",", "", regex=False))

In [ ]:
#convert column from object to string 
acs2024_gdf["tract_id"] = acs2024_gdf["tract_id"].astype(str)

In [ ]:
acs2024_gdf.head()

In [ ]:
#import 2019 acs data
acs2019_df = pd.read_csv("ACSDT5Y2019.B25064-2026-04-30T132432.csv")

In [ ]:
#transpose the data
acs2019_df = acs2019_df.transpose().reset_index()

In [ ]:
#fix column names 
acs2019_df.columns = acs2019_df.iloc[0]


In [ ]:
#drop first row (column names)
acs2019_df = acs2019_df.drop(acs2019_df.index[0]).reset_index(drop=True)

In [ ]:
#rename columns
acs2019_df = acs2019_df.rename(columns={"Label (Grouping)": "tract_name", "Median gross rent": "median_rent_2019"})

In [ ]:
#create a column with just the tract number
acs2019_df["tract_id"] = (acs2019_df["tract_name"].str.split().str[2].str.replace(",", "", regex=False))

In [ ]:
#convert column from object to string 
acs2019_df["tract_id"] = acs2019_df["tract_id"].astype(str)

In [ ]:
#break up the estimate and se rows
acs2019_df = acs2019_df.drop_duplicates(subset="tract_id", keep="first").reset_index(drop=True)

In [ ]:
acs2019_df.head()